## 1. Sample sheets and files

In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/mahdyar/uniproject")

In [8]:
tumor_samples = pd.read_csv(ROOT / "Data" / "gdc_sample_sheet.2026-09-28.tsv", sep="\t")
normal_samples = pd.read_csv(
    ROOT / "DataNormal" / "gdc_sample_sheet.normal.2026-10-13.tsv", sep="\t"
)
sample_sheet = pd.concat([tumor_samples, normal_samples], ignore_index=True)
sample_sheet.head()

,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
0,ead53b27-6ad9-4b96-b5d4-0d4f06fb2d13,dadd1789-354a-4abb-8d4e-8556166cf341.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-A8-A09E,TCGA-A8-A09E-01A,Tumor,Primary,Unknown,Unknown
1,0e1999e0-9c02-4804-88b6-60b46d100739,db5dab56-838e-4784-a7cf-f05296e356f7.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-A7-A0DC,TCGA-A7-A0DC-01A,Tumor,Primary,Unknown,Unknown
2,3e8cff80-ed17-479c-8327-ca8e71ee7482,0c814cd9-5b2c-4764-9e6a-6ad0403fcd7d.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-A7-A0DC,TCGA-A7-A0DC-01B,Tumor,Primary,Unknown,FFPE
3,385ddd6f-c126-4c4f-acde-1ef2d74d3639,616b6b60-8c4a-4d85-9a24-f47df6ad497e.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-A8-A08F,TCGA-A8-A08F-01A,Tumor,Primary,Unknown,Unknown
4,91060c43-f1e3-49ea-a3c9-96def417716b,caf3d917-623d-4e2a-ac0a-521415385763.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-A8-A091,TCGA-A8-A091-01A,Tumor,Primary,Unknown,Unknown


In [9]:
tissue_type_counts = (
    sample_sheet["Tissue Type"]
    .value_counts()
    .reset_index(name="Count")
)
tissue_type_counts

,Tissue Type,Count
0,Tumor,1111
1,Normal,113


In [10]:
clinical_data = pd.read_json(ROOT / "Data" / "clinical.cart.2026-09-28.json")
clinical_data.head()

,disease_type,project,submitter_id,days_to_consent,diagnoses,consent_type,demographic,primary_site,updated_datetime,case_id,follow_ups,index_date,state,lost_to_followup,exposures
0,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-E2-A1IU,-34.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': 'd14426b2-e0a0-519a-bea6-4f...,Breast,2025-01-06T00:20:17.681998-06:00,001cef41-ff86-4d3f-a140-a647ac4b10a1,"[{'timepoint_category': 'Last Contact', 'follo...",Diagnosis,released,NaN,NaN
1,Adenomas and Adenocarcinomas,{'project_id': 'TCGA-BRCA'},TCGA-A1-A0SB,76.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': 'fa693617-eab9-502c-9a16-0e...,Breast,2025-01-05T21:27:11.301925-06:00,0045349c-69d9-4306-a403-c9c1fa836644,"[{'timepoint_category': 'Last Contact', 'follo...",Diagnosis,released,NaN,NaN
2,Adnexal and Skin Appendage Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-A2-A04W,19.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': '6562e53d-99fd-54ed-b8e1-51...,Breast,2025-01-06T07:31:06.617417-06:00,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,No,NaN
3,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-AN-A0AM,0.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': '7366952a-e8e7-56ec-9867-23...,Breast,2025-01-05T22:52:03.042081-06:00,00a2d166-78c9-4687-a195-3d6315c27574,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,Yes,NaN
4,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-LL-A440,11.0,"[{'ajcc_pathologic_t': 'Tis (DCIS)', 'morpholo...",Informed Consent,{'demographic_id': 'f5229922-62e2-51d5-ba4e-94...,Breast,2025-01-05T22:47:37.828549-06:00,00b11ca8-8540-4a3d-b602-ec754b00230b,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,No,NaN


In [11]:
# Count distinct tumor files for each patient (Case ID).
tumor_files_per_patient = (
    sample_sheet.loc[sample_sheet["Tissue Type"] == "Tumor"]
    .groupby("Case ID")["File ID"]
    .nunique()
)

# Count how many patients have each number of tumor files.
tumor_file_counts = (
    tumor_files_per_patient.value_counts()
    .sort_index()
    .rename_axis("Tumor files per patient")
    .reset_index(name="Number of Patients")
)
tumor_file_counts

,Tumor files per patient,Number of Patients
0,1,1084
1,2,6
2,3,5


In [12]:
# open a gene exression file and check the first few rows
gene_expression_file = ROOT / "Data" / "70ea9273-29f3-425a-ba66-5b75db62c9b7" / "8f6c4058-6bdd-4f85-94ed-874966ffec55.rna_seq.augmented_star_gene_counts.tsv"
gene_expression_data = pd.read_csv(gene_expression_file, sep="\t")
gene_expression_data.head(10)

,,,,,,,,# gene-model: GENCODE v36
gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
N_unmapped,nan,nan,2540980,2540980,2540980,nan,nan,NaN
N_multimapping,nan,nan,7838441,7838441,7838441,nan,nan,NaN
N_noFeature,nan,nan,2818084,39142825,39260267,nan,nan,NaN
N_ambiguous,nan,nan,8917223,2213548,2211601,nan,nan,NaN
ENSG00000000003.15,TSPAN6,protein_coding,17607,8735,8872,148.1058,60.6736,67.9268
ENSG00000000005.6,TNMD,protein_coding,73,29,44,1.8871,0.7731,0.8655
ENSG00000000419.13,DPM1,protein_coding,3197,1574,1623,101.0637,41.4021,46.3515
ENSG00000000457.14,SCYL3,protein_coding,1220,1315,1305,6.7630,2.7706,3.1018
ENSG00000000460.17,C1orf112,protein_coding,440,971,958,2.8121,1.1520,1.2898


## 2. Labels available in the data

**There are no PAM50 subtype labels in these files.** The GDC download (sample sheets, clinical JSON, biospecimen JSON) has no PAM50 or intrinsic-subtype field. To work on subtypes you need a published label table for TCGA-BRCA and must join it to these samples by TCGA barcode. One common source is the TCGA PanCancer Atlas subtype calls, which cBioPortal shows as `SUBTYPE` (BRCA_LumA, BRCA_LumB, BRCA_Her2, BRCA_Basal, BRCA_Normal).

These are the labels you **do** have, one value per patient (case):

| Label | Where it comes from | What it means |
| --- | --- | --- |
| **Tissue type** (Tumor / Normal) | Sample sheets | Tumor = primary breast tumour sample (barcode `-01`). Normal = adjacent normal breast tissue from the same patients (barcode `-11`). This is about the *sample*, not a cancer subtype. Shown in section 1. |
| **Histology** (`primary_diagnosis`) | Clinical JSON, diagnoses | The pathologist's diagnosis of the tumour's microscopic type. *Infiltrating duct carcinoma* (IDC) starts in the milk ducts and is the most common type. *Lobular carcinoma* (ILC) starts in the milk-producing lobules and grows in single-file lines, because it usually lacks the adhesion protein E-cadherin. Others are mixed or rare special types. This was the original project target. |
| **ER / PR status** | Clinical JSON, follow-ups, molecular tests (IHC) | Whether tumour cells have oestrogen (ER, gene *ESR1*) or progesterone (PR, gene *PGR*) receptors, measured by immunohistochemistry (antibody staining). Positive tumours are usually driven by hormones and treated with hormone therapy. |
| **HER2 status** | Same place (IHC and FISH) | Whether the *ERBB2* gene is over-active. IHC scores the protein (0, 1+, 2+, 3+). FISH counts gene copies and settles the 2+ "equivocal" cases, so FISH is used here when present. HER2-positive tumours are treated with drugs like trastuzumab. |
| **Receptor group** | Derived from ER, PR, HER2 | Clinical shorthand: *HR+/HER2−*, *HER2+* and *triple-negative* (ER−, PR−, HER2−). These **roughly** track PAM50 (Luminal A/B ≈ HR+, HER2-enriched ≈ HER2+, Basal-like ≈ triple-negative) but they are a different, protein-based classification and are **not** PAM50 labels. |
| **AJCC stage** and **T / N / M** | Clinical JSON, diagnoses | How far the cancer had spread at diagnosis. T = tumour size, N = lymph-node involvement, M = distant metastasis (M1 = yes). Stage I to IV combines them. |
| **Vital status** | Clinical JSON, demographic | Alive or Dead at last contact. Together with follow-up times it is used for survival analysis, not classification. |

Rules used below: each patient's *primary* diagnosis record is used (some patients also have records for metastases, recurrences or other cancers). For each receptor, any Positive result counts as Positive, otherwise Negative, otherwise Equivocal. Counts are for the 1,095 patients in the tumour sample sheet.

In [ ]:
import numpy as np

tumor_cases = set(tumor_samples["Case ID"])
clinical = clinical_data[clinical_data["submitter_id"].isin(tumor_cases)]


def primary_diagnosis_record(diagnoses):
    """Return the diagnosis record for the primary tumour, or {} if there is none."""
    if not isinstance(diagnoses, list) or not diagnoses:
        return {}
    for d in diagnoses:
        if d.get("classification_of_tumor") == "primary":
            return d
    return diagnoses[0] if len(diagnoses) == 1 else {}


def histology_group(diagnosis):
    if diagnosis == "Infiltrating duct carcinoma, NOS":
        return "Ductal (IDC)"
    if diagnosis == "Lobular carcinoma, NOS":
        return "Lobular (ILC)"
    if isinstance(diagnosis, str) and ("duct" in diagnosis.lower() or "lobular" in diagnosis.lower()):
        return "Mixed ductal/lobular or other"
    if isinstance(diagnosis, str) and diagnosis != "Not Reported":
        return "Other special type"
    return "Missing"


def receptor_status(follow_ups, gene, prefer_fish=False):
    """Combine all test results for one gene into Positive / Negative / Equivocal / Missing."""
    results = {"IHC": [], "FISH": []}
    for fu in follow_ups if isinstance(follow_ups, list) else []:
        for t in fu.get("molecular_tests", []):
            if t.get("gene_symbol") == gene and t.get("test_result") in ("Positive", "Negative", "Equivocal"):
                results.setdefault(t.get("molecular_analysis_method"), []).append(t["test_result"])
    tests = results["FISH"] if prefer_fish and results["FISH"] else sum(results.values(), [])
    for status in ("Positive", "Negative", "Equivocal"):
        if status in tests:
            return status
    return "Missing"


rows = []
for _, case in clinical.iterrows():
    dx = primary_diagnosis_record(case["diagnoses"])
    demo = case["demographic"] if isinstance(case["demographic"], dict) else {}
    rows.append({
        "case_id": case["submitter_id"],
        "primary_diagnosis": dx.get("primary_diagnosis", "Missing"),
        "histology": histology_group(dx.get("primary_diagnosis")),
        "ER": receptor_status(case["follow_ups"], "ESR1"),
        "PR": receptor_status(case["follow_ups"], "PGR"),
        "HER2": receptor_status(case["follow_ups"], "ERBB2", prefer_fish=True),
        "stage": dx.get("ajcc_pathologic_stage", "Missing"),
        "T": dx.get("ajcc_pathologic_t", "Missing"),
        "N": dx.get("ajcc_pathologic_n", "Missing"),
        "M": dx.get("ajcc_pathologic_m", "Missing"),
        "vital_status": demo.get("vital_status", "Missing"),
    })
labels = pd.DataFrame(rows).set_index("case_id")


def receptor_group(r):
    if "Missing" in (r.ER, r.PR, r.HER2) or "Equivocal" in (r.ER, r.PR, r.HER2):
        return "Unknown"
    if r.HER2 == "Positive":
        return "HER2+"
    if "Positive" in (r.ER, r.PR):
        return "HR+/HER2-"
    return "Triple-negative"


labels["receptor_group"] = labels.apply(receptor_group, axis=1)
print(f"{len(labels)} patients")
labels.head()

,disease_type,project,submitter_id,days_to_consent,diagnoses,consent_type,demographic,primary_site,updated_datetime,case_id,follow_ups,index_date,state,lost_to_followup,exposures
0,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-E2-A1IU,-34.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': 'd14426b2-e0a0-519a-bea6-4f...,Breast,2025-01-06T00:20:17.681998-06:00,001cef41-ff86-4d3f-a140-a647ac4b10a1,"[{'timepoint_category': 'Last Contact', 'follo...",Diagnosis,released,NaN,NaN
1,Adenomas and Adenocarcinomas,{'project_id': 'TCGA-BRCA'},TCGA-A1-A0SB,76.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': 'fa693617-eab9-502c-9a16-0e...,Breast,2025-01-05T21:27:11.301925-06:00,0045349c-69d9-4306-a403-c9c1fa836644,"[{'timepoint_category': 'Last Contact', 'follo...",Diagnosis,released,NaN,NaN
2,Adnexal and Skin Appendage Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-A2-A04W,19.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': '6562e53d-99fd-54ed-b8e1-51...,Breast,2025-01-06T07:31:06.617417-06:00,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,No,NaN
3,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-AN-A0AM,0.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': '7366952a-e8e7-56ec-9867-23...,Breast,2025-01-05T22:52:03.042081-06:00,00a2d166-78c9-4687-a195-3d6315c27574,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,Yes,NaN
4,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-LL-A440,11.0,"[{'ajcc_pathologic_t': 'Tis (DCIS)', 'morpholo...",Informed Consent,{'demographic_id': 'f5229922-62e2-51d5-ba4e-94...,Breast,2025-01-05T22:47:37.828549-06:00,00b11ca8-8540-4a3d-b602-ec754b00230b,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,No,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-A2-A0CP,0.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': 'bb8d0641-2529-5ec3-838a-3c...,Breast,2025-01-06T07:32:51.280927-06:00,fe15c48b-116d-461e-9b40-68514730fd18,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,NaN,NaN
1091,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-PL-A8LX,-50.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': '562fee57-d81f-52ce-8b4e-22...,Breast,2025-01-05T23:46:38.629779-06:00,fe2cd610-aa52-4789-ac62-7683281bb22f,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,Yes,NaN
1092,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-A2-A3XZ,49.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': '950c1911-2466-5e9c-9953-76...,Breast,2025-01-05T22:42:16.997018-06:00,fe7f74b8-20f4-4471-91dc-4cca8c68e5c0,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,No,NaN
1093,Ductal and Lobular Neoplasms,{'project_id': 'TCGA-BRCA'},TCGA-E9-A295,-1.0,"[{'synchronous_malignancy': 'No', 'ajcc_pathol...",Informed Consent,{'demographic_id': 'dc38f171-af3e-5f95-8c74-0d...,Breast,2025-01-05T23:26:38.867563-06:00,fec0da58-1047-44d2-b6d1-c18cceed43dc,"[{'timepoint_category': 'Follow-up', 'follow_u...",Diagnosis,released,Yes,NaN


In [15]:
def count_table(column):
    counts = labels[column].value_counts(dropna=False)
    return pd.DataFrame({"Patients": counts, "Percent": (100 * counts / counts.sum()).round(1)})

### Histology

In [16]:
count_table("histology")

,Patients,Percent
histology,,
Ductal (IDC),777,71.0
Lobular (ILC),201,18.4
Mixed ductal/lobular or other,62,5.7
Other special type,54,4.9
Missing,1,0.1


In [17]:
# The detailed diagnosis behind the groups above
count_table("primary_diagnosis")

,Patients,Percent
primary_diagnosis,,
"Infiltrating duct carcinoma, NOS",777,71.0
"Lobular carcinoma, NOS",201,18.4
Infiltrating duct and lobular carcinoma,28,2.6
Infiltrating duct mixed with other types of carcinoma,19,1.7
Mucinous adenocarcinoma,16,1.5
"Metaplastic carcinoma, NOS",14,1.3
Infiltrating lobular mixed with other types of carcinoma,6,0.5
Intraductal papillary adenocarcinoma with invasion,6,0.5
"Medullary carcinoma, NOS",6,0.5


### Receptor status (ER, PR, HER2) and receptor group

In [18]:
pd.concat({c: labels[c].value_counts() for c in ["ER", "PR", "HER2"]}, axis=1).fillna(0).astype(int)

,ER,PR,HER2
Positive,808,699,182
Negative,236,342,768
Missing,49,50,114
Equivocal,2,4,31


In [19]:
count_table("receptor_group")

,Patients,Percent
receptor_group,,
HR+/HER2-,605,55.3
HER2+,181,16.5
Triple-negative,159,14.5
Unknown,150,13.7


### PAM50 molecular subtype

PAM50 calls are not in the GDC files. They come from `Labels/pam50_tcga_pancan_atlas_2018.tsv` (TCGA PanCancer Atlas calls from cBioPortal, one row per patient; see `Labels/README.md`) and are joined on the TCGA case barcode. Patients without a call are counted as Missing.

In [20]:
pam50 = pd.read_csv(ROOT / "Labels" / "pam50_tcga_pancan_atlas_2018.tsv", sep="\t", index_col="case_id")
pam50_names = {
    "LumA": "Luminal A",
    "LumB": "Luminal B",
    "Her2": "HER2-enriched",
    "Basal": "Basal-like",
    "Normal": "Normal-like",
}
labels["pam50"] = pam50["pam50"].map(pam50_names).reindex(labels.index).fillna("Missing")
count_table("pam50")

,Patients,Percent
pam50,,
Luminal A,499,45.6
Luminal B,197,18.0
Basal-like,171,15.6
Missing,114,10.4
HER2-enriched,78,7.1
Normal-like,36,3.3


### Stage and vital status

In [21]:
count_table("stage")

,Patients,Percent
stage,,
Stage IIA,358,32.7
Stage IIB,255,23.3
Stage IIIA,155,14.2
Stage I,90,8.2
Stage IA,86,7.9
Stage IIIC,65,5.9
Stage IIIB,27,2.5
Stage IV,20,1.8
Stage X,13,1.2


In [22]:
count_table("vital_status")

,Patients,Percent
vital_status,,
Alive,942,86.0
Dead,152,13.9
Missing,1,0.1
